# IRIS Classification Pipeline on Vertex AI Workbench

This notebook demonstrates an end-to-end machine learning workflow on Google Cloud Platform.

The pipeline includes:

1. Loading the IRIS dataset
2. Uploading data to Google Cloud Storage (GCS)
3. Training a Decision Tree Classifier
4. Saving model artifacts
5. Storing artifacts in GCS
6. Running inference using a separate script
7. Organizing outputs by execution timestamp

This implementation satisfies Tasks 2–5 of the MLOps Week 1 Assignment.

In [56]:
import sys
print(sys.executable)

import sklearn
print("sklearn OK")

/opt/micromamba/envs/jupyterlab/bin/python3
sklearn OK


# Step 1: Configure GCP Project

In [58]:
! pip3 install --upgrade --quiet  google-cloud-aiplatform

In [59]:
PROJECT_ID = "project-ad35ebae-2568-47b9-83a"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

In [60]:
BUCKET_URI = f"gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique"

In [61]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

In [62]:
import os
import sys

In [63]:
import sys
print(sys.executable)

/opt/micromamba/envs/jupyterlab/bin/python3


In [64]:
import sys

!{sys.executable} -m pip install \
    scikit-learn \
    pandas \
    numpy \
    matplotlib \
    google-cloud-aiplatform

In [65]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

data = pd.read_csv('data/iris.csv')
data.head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [66]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [67]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


In [68]:
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

In [69]:
MODEL_ARTIFACT_DIR = "iris_classifier/model"

In [70]:
!gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


In [71]:
!ls -lah artifacts

total 16K
drwxr-xr-x  3 jupyter jupyter 4.0K Jun 21 17:08 .
drwxr-xr-x 14 jupyter jupyter 4.0K Jun 21 17:07 ..
drwxr-xr-x  2 jupyter jupyter 4.0K Jun 21 16:57 .ipynb_checkpoints
-rw-r--r--  1 jupyter jupyter 2.6K Jun 21 17:08 model.joblib


In [72]:
!gcloud storage buckets list

---
creation_time: 2026-06-21T11:06:28+0000
default_storage_class: STANDARD
generation: 1782039988057847246
location: US-CENTRAL1
location_type: region
metageneration: 1
name: mlops-course-project-ad35ebae-2568-47b9-83a-unique
public_access_prevention: inherited
soft_delete_policy:
  effectiveTime: '2026-06-21T11:06:28.436000+00:00'
  retentionDurationSeconds: '604800'
storage_url: gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/
uniform_bucket_level_access: true
update_time: 2026-06-21T11:06:28+0000


In [73]:
print(BUCKET_URI)

gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique


In [74]:
!gsutil ls -r {BUCKET_URI}

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/iris_classifier/:

gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/iris_classifier/model/:
gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/iris_classifier/model/model.joblib


In [75]:
!which python

/opt/micromamba/bin/python


In [76]:
import sys
print(sys.executable)

/opt/micromamba/envs/jupyterlab/bin/python3


In [77]:
from datetime import datetime

RUN1 = datetime.now().strftime("%Y%m%d_%H%M%S")

print(RUN1)

20260621_172017


In [78]:
RUN1_MODEL_DIR = f"runs/{RUN1}/model"

!gsutil cp artifacts/model.joblib \
{BUCKET_URI}/{RUN1_MODEL_DIR}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


In [79]:
!gsutil ls -r {BUCKET_URI}/runs/**

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/runs/20260621_172017/model/model.joblib


In [80]:
!/opt/micromamba/envs/jupyterlab/bin/python3 inference.py \
--bucket mlops-course-project-ad35ebae-2568-47b9-83a-unique \
--model_path runs/{RUN1}/model/model.joblib \
--data_path data/v1.csv \
--run_id {RUN1}

Model loaded from GCS
Accuracy = 0.9512195121951219
Metrics uploaded


In [81]:
!gsutil ls -r {BUCKET_URI}/runs/**

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/runs/20260621_172017/metrics/metrics.json
gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/runs/20260621_172017/model/model.joblib


In [82]:
from datetime import datetime

RUN2 = datetime.now().strftime("%Y%m%d_%H%M%S")

In [83]:
RUN2_MODEL_DIR = f"runs/{RUN2}/model"

!gsutil cp artifacts/model.joblib \
{BUCKET_URI}/{RUN2_MODEL_DIR}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


In [84]:
!/opt/micromamba/envs/jupyterlab/bin/python3 inference.py \
--bucket mlops-course-project-ad35ebae-2568-47b9-83a-unique \
--model_path runs/{RUN2}/model/model.joblib \
--data_path data/v2.csv \
--run_id {RUN2}

Model loaded from GCS
Accuracy = 1.0
Metrics uploaded


In [85]:
from google.cloud import storage
import json

client = storage.Client()

bucket = client.bucket(
    "mlops-course-project-ad35ebae-2568-47b9-83a-unique"
)

for run in [RUN1, RUN2]:

    blob = bucket.blob(
        f"runs/{run}/metrics/metrics.json"
    )

    print(blob.download_as_text())

{
  "run_id": "20260621_172017",
  "dataset": "data/v1.csv",
  "accuracy": 0.9512195121951219
}
{
  "run_id": "20260621_172249",
  "dataset": "data/v2.csv",
  "accuracy": 1.0
}


In [8]:
!gcloud storage buckets list

---
creation_time: 2026-06-21T11:06:28+0000
default_storage_class: STANDARD
generation: 1782039988057847246
location: US-CENTRAL1
location_type: region
metageneration: 1
name: mlops-course-project-ad35ebae-2568-47b9-83a-unique
public_access_prevention: inherited
soft_delete_policy:
  effectiveTime: '2026-06-21T11:06:28.436000+00:00'
  retentionDurationSeconds: '604800'
storage_url: gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/
uniform_bucket_level_access: true
update_time: 2026-06-21T11:06:28+0000


## WEEK 1 MLops Task ( 1- 6 )

# Step 2: Upload Dataset to Google Cloud Storage


Machine Learning pipelines should not depend on local files.

To make the pipeline reproducible and cloud-native, we store the IRIS dataset inside a Google Cloud Storage bucket.

This bucket will serve as the central storage location for:

- Input datasets
- Trained models
- Metrics
- Logs

In [86]:
import pandas as pd
import joblib
import json

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [87]:
PROJECT_ID = "project-ad35ebae-2568-4769-83a"

BUCKET_URI = "gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique"

# Step 3: Load Dataset

We load the IRIS dataset and inspect its structure.

In [88]:
!gsutil cp data/iris.csv {BUCKET_URI}/data/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://data/iris.csv [Content-Type=text/csv]...
/ [1 files][  3.8 KiB/  3.8 KiB]                                                
Operation completed over 1 objects/3.8 KiB.                                      


In [89]:
!gsutil ls {BUCKET_URI}/data/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://mlops-course-project-ad35ebae-2568-47b9-83a-unique/data/iris.csv


In [90]:
data = pd.read_csv("data/iris.csv")

data.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


The dataset contains:

- Sepal Length
- Sepal Width
- Petal Length
- Petal Width

Target Variable:

- Species

# Step 4: Split Dataset

The dataset is divided into:

- Training Set (60%)
- Test Set (40%)

Stratified sampling is used to preserve class distribution across both datasets.

The test set will later be used for model evaluation.

In [91]:
train, test = train_test_split(
    data,
    test_size=0.4,
    stratify=data["species"],
    random_state=42
)

X_train = train[
    ["sepal_length","sepal_width","petal_length","petal_width"]
]

y_train = train["species"]

X_test = test[
    ["sepal_length","sepal_width","petal_length","petal_width"]
]

y_test = test["species"]

In [92]:
model = DecisionTreeClassifier(
    max_depth=3,
    random_state=1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Accuracy:", round(acc,4))

Accuracy: 0.9833


# Step 7: Save Model Artifact

The trained model is serialized using Joblib.

This produces:

artifacts/model.joblib

This file represents the trained model and can later be reused for inference without retraining.

In [93]:
import os

os.makedirs("artifacts", exist_ok=True)

joblib.dump(
    model,
    "artifacts/model.joblib"
)

['artifacts/model.joblib']

In [94]:
RUN1 = datetime.now().strftime("%Y%m%d_%H%M%S")

print(RUN1)

20260621_173033


# Step 8: Create Execution Timestamp

Every pipeline execution receives a unique timestamp.

Benefits:

- Traceability
- Experiment Tracking
- Reproducibility

Example:

20260621_134500

# Step 9: Store Model Artifact in GCS

The trained model is uploaded to Google Cloud Storage.

Folder structure:

runs/<timestamp>/model/

This allows multiple training runs to coexist without overwriting previous results.

In [95]:
MODEL_ARTIFACT_DIR = f"runs/{RUN1}/model"

!gsutil cp artifacts/model.joblib \
{BUCKET_URI}/{MODEL_ARTIFACT_DIR}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


In [96]:
!/opt/micromamba/envs/jupyterlab/bin/python3 inference.py \
--bucket mlops-course-project-ad35ebae-2568-47b9-83a-unique \
--model_path runs/{RUN1}/model/model.joblib \
--data_path data/iris.csv \
--run_id {RUN1}

Model fetched from GCS
Accuracy: 0.9833333333333333
Metrics uploaded to GCS


In [97]:
RUN2 = datetime.now().strftime("%Y%m%d_%H%M%S")

print(RUN2)

20260621_173205


In [98]:
MODEL_ARTIFACT_DIR = f"runs/{RUN2}/model"

!gsutil cp artifacts/model.joblib \
{BUCKET_URI}/{MODEL_ARTIFACT_DIR}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


# Step 11: Create Inference Script

Inference is separated from training.

This follows MLOps best practices.

The inference script:

1. Fetches model artifacts from GCS
2. Loads the trained model
3. Runs predictions
4. Computes evaluation metrics
5. Stores results back in GCS

This satisfies the assignment requirement of separating training and inference.

In [99]:
!/opt/micromamba/envs/jupyterlab/bin/python3 inference.py \
--bucket mlops-course-project-ad35ebae-2568-47b9-83a-unique \
--model_path runs/{RUN2}/model/model.joblib \
--data_path data/iris.csv \
--run_id {RUN2}

Model fetched from GCS
Accuracy: 0.9833333333333333
Metrics uploaded to GCS


# Step 14: Multiple Data Versions (Optional)

Two dataset versions are evaluated:

- data/v1.csv
- data/v2.csv

The same trained model is used to generate predictions on both versions.

Metrics are compared to understand how data changes affect model performance.

In [100]:
!/opt/micromamba/envs/jupyterlab/bin/python3 inference.py \
--bucket mlops-course-project-ad35ebae-2568-47b9-83a-unique \
--model_path runs/{RUN2}/model/model.joblib \
--data_path data/v1.csv \
--run_id {RUN2}_v1

Model fetched from GCS
Accuracy: 0.9512195121951219
Metrics uploaded to GCS


In [101]:
!/opt/micromamba/envs/jupyterlab/bin/python3 inference.py \
--bucket mlops-course-project-ad35ebae-2568-47b9-83a-unique \
--model_path runs/{RUN2}/model/model.joblib \
--data_path data/v2.csv \
--run_id {RUN2}_v2

Model fetched from GCS
Accuracy: 1.0
Metrics uploaded to GCS
